In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ==========================================
# PART 1: TEXT GENERATION MODELS
# ==========================================

class VanillaRNN(nn.Module):
    """
    Task 1: Implement the basic RNN formula.
    Refer to Lecture Slide 15.
    """
    def __init__(self, input_size, hidden_size, output_size):
        super(VanillaRNN, self).__init__()
        self.hidden_size = hidden_size

        # TODO 1.1: Define the linear layers.
        # You need to transform the combined input (input + hidden) into a new hidden state.
        # You also need to transform the combined input into the output.
        # Hint: The size of the combined input is input_size + hidden_size.
        # Linear layer to compute the next hidden state
        self.i2h = nn.Linear(input_size +hidden_size, hidden_size)
        
        # Linear layer to compute output logits
        self.i2o = nn.Linear(input_size + hidden_size,output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, input_tensor, hidden_tensor):
        """
        input_tensor shape: (1, input_size)
        hidden_tensor shape: (1, hidden_size)
        """
        # TODO 1.2: Concatenate input_tensor and hidden_tensor along dimension 1 using torch.cat
        # Concatenate input and previous hidden state
        combined = torch.cat((input_tensor,hidden_tensor),dim=1)

        # TODO 1.3: Calculate the new hidden state.
        # Apply the linear layer self.i2h to 'combined', then apply torch.tanh
        # Compute new hidden state using tanh activation
        hidden = torch.tanh(self.i2h(combined))

        # TODO 1.4: Calculate the output.
        # Apply the linear layer self.i2o to 'combined', then apply self.softmax
        output = self.softmax(self.i2o(combined))

        return output, hidden

    def initHidden(self):
        return torch.zeros(1, self.hidden_size)


class LSTMGenerator(nn.Module):
    """
    Task 2: Use PyTorch's built-in LSTM layer.
    """
    def __init__(self, input_size, hidden_size, output_size):
        super(LSTMGenerator, self).__init__()
        self.hidden_size = hidden_size

        # TODO 2.1: Define the LSTM layer.
        # Use nn.LSTM(input_size, hidden_size)
        # input_size -> hidden_size
        self.lstm = nn.LSTM(input_size, hidden_size)

        # TODO 2.2: Define the output layer.
        # Map from hidden_size to output_size
        self.out = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, input_tensor, hidden_tuple):
        # TODO 2.3: Reshape input_tensor.
        # nn.LSTM expects input of shape (seq_len, batch, input_size).
        # Since we do 1 char at a time, seq_len=1, batch=1.
        # Use input_tensor.view(...)
        input_view = input_tensor.view(1,1,-1)

        # TODO 2.4: Pass through LSTM layer.
        # output, hidden_tuple = self.lstm(...)
        output, hidden_tuple = self.lstm(input_view, hidden_tuple)

        # TODO 2.5: Project output to vocabulary size.
        # The output from LSTM is (1, 1, hidden_size). Access the first element [0].
        prediction = self.out(output[0])
        
        output = self.softmax(prediction)
        
        return output, hidden_tuple

    def initHidden(self):
        # LSTM state is a tuple (h_0, c_0)
        return (torch.zeros(1, 1, self.hidden_size), torch.zeros(1, 1, self.hidden_size))


# ==========================================
# PART 2: NEURAL TRANSLATION MODELS
# ==========================================

class EncoderRNN(nn.Module):
    """
    (PROVIDED) Simple Encoder.
    Students do not need to modify this.
    """
    def __init__(self, input_size, hidden_size):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size)

    def forward(self, input_seq, hidden):
        embedded = self.embedding(input_seq).view(1, 1, -1)
        output, hidden = self.lstm(embedded, hidden)
        return output, hidden

    def initHidden(self):
        return (torch.zeros(1, 1, self.hidden_size), torch.zeros(1, 1, self.hidden_size))


class BahdanauAttention(nn.Module):
    """
    Task 3.1: Implement the Attention Mechanism.
    Refer to Lecture Slide 84-86.
    Formula: score = V * tanh(Wa(keys) + Ua(query))
    """
    def __init__(self, hidden_size):
        super(BahdanauAttention, self).__init__()
        self.hidden_size = hidden_size

        # TODO 3.1.1: Define the weights for the attention score. You will use the function of nn.Linear( , ).
        # Wa: Linear layer to project keys (encoder_outputs) -> hidden_size
        # Ua: Linear layer to project query (decoder_hidden) -> hidden_size
        # Va: Linear layer to project the tanh result -> 1 (scalar energy score)
        
        # Projects encoder 
        self.Wa = nn.Linear(hidden_size,hidden_size)
        # Projects decoder
        self.Ua = nn.Linear(hidden_size, hidden_size)

        # Produces scalar energy score
        self.Va = nn.Linear(hidden_size,1)

    def forward(self, query, keys):
        """
        query: current decoder hidden state (1, 1, hidden_size)
        keys: all encoder outputs (seq_len, 1, hidden_size)
        """

        # Calculate Energy Scores.
        # 1. Project keys using self.Wa
        # 2. Project query using self.Ua
        # 3. Add them (PyTorch broadcasting handles the shape mismatch)
        # 4. Apply torch.tanh
        # 5. Project result using self.Va
        scores = self.Va(torch.tanh(self.Wa(keys) + self.Ua(query)))

        # Calculate Attention Weights (alpha).
        # Apply F.softmax to scores. Be careful with dimensions!
        # The shape of scores will be (seq_len, 1, 1). You want softmax over seq_len (dim=0) or dim=1 after squeezing.
        # Target shape: (1, seq_len)
        scores = scores.squeeze(2).permute(1, 0)
        weights = F.softmax(scores, dim=1)

        # Calculate Context Vector.
        # Compute the weighted sum of keys: sum(weights * keys).
        # Hint: Use torch.bmm (batch matrix multiplication).
        # Reshape weights to (1, 1, seq_len) and keys to (1, seq_len, hidden) for bmm.
        context = torch.bmm(weights.unsqueeze(0), keys.permute(1, 0, 2))

        return context, weights


class AttnDecoderRNN(nn.Module):
    """
    Task 3.2: Decoder with Attention.
    """
    def __init__(self, hidden_size, output_size, dropout_p=0.1):
        super(AttnDecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.dropout_p = dropout_p

        self.embedding = nn.Embedding(self.output_size, self.hidden_size)
        self.attention = BahdanauAttention(self.hidden_size)

        # TODO 3.2.1: Define LSTM layer.
        # The input to the LSTM will be the concatenation of the
        # embedded input word (size: hidden_size) AND the context vector (size: hidden_size).
        # So, input_size for LSTM = hidden_size * 2
        self.lstm = nn.LSTM(hidden_size * 2, hidden_size)

        
        self.out = nn.Linear(self.hidden_size, self.output_size)

    def forward(self, input_step, hidden, encoder_outputs):
        # 1. Get embedding of current input word
        embedded = self.embedding(input_step).view(1, 1, -1)
        embedded = F.dropout(embedded, self.dropout_p)

        # TODO 3.2.2: Calculate Attention.
        # Use self.attention(query, keys).
        # Query is hidden[0] (the hidden state h_n, ignoring cell state c_n).
        # Keys are encoder_outputs.
        context, attn_weights = self.attention(hidden[0],encoder_outputs)

        # TODO 3.2.3: Combine Embedding and Context.
        # Concatenate 'embedded' and 'context' along the feature dimension (dim=2).
        rnn_input = torch.cat((embedded, context),dim=2)

        # TODO 3.2.4: Pass through LSTM.
        # output, hidden = self.lstm(...)
        output, hidden = self.lstm(rnn_input,hidden)

        # 5. Predict next token
        output = F.log_softmax(self.out(output[0]), dim=1)

        return output, hidden, attn_weights

# ==========================================
# SANITY CHECK (STUDENTS: RUN THIS TO TEST YOUR CODE)
# ==========================================
if __name__ == "__main__":
    print("--- Running Sanity Checks ---")

    # 1. Test Vanilla RNN
    try:
        print("Checking VanillaRNN...", end="")
        rnn = VanillaRNN(10, 20, 10)
        if rnn.i2h is None: raise NotImplementedError("Layers not defined")
        h = rnn.initHidden()
        i = torch.randn(1, 10)
        o, h = rnn(i, h)
        if o is None: raise NotImplementedError("Forward pass not implemented")
        if o.shape == (1, 10) and h.shape == (1, 20): print(" PASS")
        else: print(" FAIL (Wrong output shapes)")
    except Exception as e:
        print(f" FAIL ({e})")

    # 2. Test LSTM
    try:
        print("Checking LSTMGenerator...", end="")
        lstm = LSTMGenerator(10, 20, 10)
        if lstm.lstm is None: raise NotImplementedError("Layers not defined")
        h = lstm.initHidden()
        i = torch.randn(1, 10)
        o, h = lstm(i, h)
        if o is None: raise NotImplementedError("Forward pass not implemented")
        if o.shape == (1, 10): print(" PASS")
        else: print(" FAIL (Wrong output shapes)")
    except Exception as e:
        print(f" FAIL ({e})")

    # 3. Test Attention
    try:
        print("Checking BahdanauAttention...", end="")
        attn = BahdanauAttention(20)
        if attn.Wa is None: raise NotImplementedError("Layers not defined")
        query = torch.randn(1, 1, 20)
        keys = torch.randn(5, 1, 20)
        c, w = attn(query, keys)
        if c is None: raise NotImplementedError("Forward pass not implemented")
        if c.shape == (1, 1, 20) and w.shape == (1, 5): print(" PASS")
        else: print(f" FAIL (Context shape: {c.shape}, Weights shape: {w.shape})")
    except Exception as e:
        print(f" FAIL ({e})")

--- Running Sanity Checks ---
Checking VanillaRNN... PASS
Checking LSTMGenerator... PASS
Checking BahdanauAttention... PASS
